In [ ]:
from torch.nn import *
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def center(a):
    return a.mul(2.0).add_(-1.0)


def dat(k, c, s=1, n=1, l=1, rx=1.0, rk=1.0, rb=1.0):
    l = 2 ** int(np.log2(c * s * (k + 1) * l))
    y = torch.randint(c * s, (n, l))
    return (
        torch.randn(n, c * s, k)
        .mul_(rx * rb * l ** (1 / k))[torch.arange(n)[:, None], y]
        .view(n * l, 1, k)
        .baddbmm(
            torch.randn(n * l, 1, k).mul_(rx),
            torch.randn(n, c * s, k, k)[torch.arange(n)[:, None], y].view(n * l, k, k),
        )
        .view(n, l, k),
        unpack(y // s if s - 1 & s else y >> int(np.log2(s)), np.log2(c).__ceil__()),
    )


class Attention(Sequential):
    def forward(s, q, k, v=None):
        return (s[0](q) @ s[1](k).transpose(-2, -1)).softmax(-1) @ s[2](k if v is None else v)


class Add(Sequential):
    def forward(s, x):
        return s[0](x[0]) + s[1](x[1])


class Batch(Module):
    @staticmethod
    def forward(x):
        return x[..., None, :, :]


class BatchLinear(Module):
    def __init__(s, i, o, n, b=None):
        super().__init__()
        s.w = Parameter(init.kaiming_uniform_(torch.empty(n, o, i)).transpose(1, 2))
        s.b = Parameter(init.uniform_(torch.empty(n, 1, o), -(i ** 0.5), i ** 0.5)) if b else None

    def forward(s, x):
        x = x @ s.w
        return x if s.b is None else x + s.b

    def extra_repr(s):
        return f"{s.w.size(1)}, {s.w.size(2)}, {s.w.size(0)}, {s.w is not None}"


class Unbatch(Module):
    @staticmethod
    def forward(x):
        return x.sum(-3)


class TrainEval(Module):
    def __init__(s, k, q, v, n=1, l=1, w=1, h=1):
        super().__init__()
        l = int(l * 2 ** v)
        w = int(w * k)
        h = int(h * w)
        L = np.log2(l).__ceil__()
        s.i = center(unpack(torch.arange(l), L))
        s.a = Attention(
            Identity(),
            Sequential(BatchLinear(k, h, n), Hardswish(1), BatchLinear(h, L, n)),
            Sequential(BatchLinear(k, h, n), Hardswish(1), *(BatchLinear(h, w, n),) if w != h else ()),
        )
        s.n = Attention(
            Sequential(BatchLinear(q, h, n), Hardswish(1), BatchLinear(h, h, n)),
            Sequential(Add(BatchLinear(w, h, n), Linear(L, h, 0)), Hardswish(1)),
            Sequential(Add(BatchLinear(w, h, n), Linear(L, h, 0)), Hardswish(1)),
        )
        s.c = Sequential(Add(BatchLinear(h, h, n), BatchLinear(q, h, n, 0)), Hardswish(1), Unbatch(), Linear(h, v))

    def forward(s, k, q):
        q = q[..., None, :, :]
        return s.c((s.n(q, (s.a(s.i, k[..., None, :, :]), s.i)), q))


class TrainEvalRdd(Module):
    def __init__(s, k, q, v, n=1, l=1, w=1, h=1):
        super().__init__()
        l = int(l * 2 ** v)
        w = int(w * k)
        h = int(h * w)
        L = np.log2(l).__ceil__()
        s.i = center(unpack(torch.arange(l), L))
        s.a = Attention(
            Sequential(BatchLinear(L, h, n), Hardswish(1), BatchLinear(h, h, n)),
            Sequential(BatchLinear(k, h, n), Hardswish(1), BatchLinear(h, h, n)),
            Sequential(BatchLinear(k, h, n), Hardswish(1), *(BatchLinear(h, w, n),) if w != h else ()),
        )
        s.b = Attention(
            Sequential(BatchLinear(q, h, n), Hardswish(1), BatchLinear(h, h, n)),
            Sequential(Add(BatchLinear(w, h, n), BatchLinear(L, h, n, 0)), Hardswish(1), BatchLinear(h, h, n)),
            Sequential(Add(BatchLinear(w, h, n), BatchLinear(L, h, n, 0)), Hardswish(1)),
        )
        s.c = Sequential(Add(BatchLinear(h, h, n), BatchLinear(q, h, n, 0)), Hardswish(1), Unbatch(), Linear(h, v))

    def forward(s, k, q):
        q = q[..., None, :, :]
        return s.c((s.b(q, (s.a(s.i, k[..., None, :, :]), s.i)), q))

In [ ]:
from os.path import exists


def get(i, j, k):
    n = 6
    m = TrainEvalRdd(2 * n, n, n, i, j, k)
    o = torch.optim.Adam(m.parameters())
    O = torch.optim.lr_scheduler.OneCycleLR(o, 0.002, 50000, pct_start=0.01, div_factor=9, final_div_factor=99)
    l = []
    for i in range(O.total_steps // 1000):
        for i in range(1000):
            x, y = dat(n, 1 << n, n=32)
            x = m(torch.cat((x, center(y)), 2), x)
            i = functional.binary_cross_entropy_with_logits(x, y)
            i.backward()
            o.step()
            O.step()
            o.zero_grad()
        x = (x.signbit() != y).all(-1)
        l.append((i.detach().cpu(), (x.sum() / x.numel()).cpu()))
    return np.array(l)


while 1:
    i = np.random.randint(1, 3), np.random.uniform(1, 2), np.random.uniform(1, 2)
    f = f"nnTrainEval2/{i[0]}{int(i[1]*1000):04}{int(i[2]*1000):04}.2f"
    if not exists(f):
        get(*i).tofile(f)